# LSTM_SNP (Lag 1)

**Dataset**: Dow Jones Industrial Index

## Description
This notebook implements a **LSTM_SNP** cell utilizing Spiking Neural P system components for time series
forecasting. Uses hard-sigmoid gates and a custom `LSTMSNPCell` underlying architecture.

- Hidden units: 8
- Single layer
- No dropout
- Stateful: hidden and cell states maintained across sequence steps

## Theory: LSTM_SNP

### LSTM Cell
Standard LSTM with forget, input, cell, and output gates.
Uses `nn.LSTM` from PyTorch with `batch_first=True`.

### Output Layer
Simple linear layer: `nn.Linear(hidden_dim, 1)` maps the last hidden state to a scalar prediction.

## Model Architecture & Implementation

In [30]:
# ============================================================
# ALL IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from math import sqrt
from scipy import stats
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PyTorch version: 2.10.0+cu128
NumPy version: 2.0.2
Using device: cuda


### LSTM_SNP (Lag 1) Cell

In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LSTMSNPCell(nn.Module):
    """
    LSTM-SNP Cell: gates r, c, o (hard sigmoid) and generated spikes a (tanh).
    u(t) = r(t)*u(t-1) - c(t)*a(t)
    h(t) = o(t)*a(t)
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.kernel = nn.Linear(input_size, hidden_size * 4, bias=False)
        self.recurrent_kernel = nn.Linear(hidden_size, hidden_size * 4, bias=False)
        self.bias = nn.Parameter(torch.zeros(hidden_size * 4))

        nn.init.xavier_uniform_(self.kernel.weight)
        nn.init.orthogonal_(self.recurrent_kernel.weight)
        
        # Initialize consumption gate bias to 1.0 (Eq requirement)
        with torch.no_grad():
            self.bias.data[hidden_size:2 * hidden_size] = 1.0

    def hard_sigmoid(self, x):
        return torch.clamp(0.2 * x + 0.5, 0.0, 1.0)

    def forward(self, x, u_tm1):
        z = self.kernel(x) + self.recurrent_kernel(u_tm1) + self.bias
        z0, z1, z2, z3 = z.chunk(4, dim=-1)

        r = self.hard_sigmoid(z0)   # reset
        c = self.hard_sigmoid(z1)   # consumption
        o = self.hard_sigmoid(z2)   # output/generation
        a = torch.tanh(z3)          # generated spikes

        u = r * u_tm1 - c * a
        h = o * a
        return h, u


class SequenceLSTMSNP(nn.Module):
    """
    LSTM-SNP for time series forecasting over a sequence.
    Stateful: maintains hidden state across sequence steps.
    """
    def __init__(self, input_dim=1, hidden_dim=8, output_dim=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.cell = LSTMSNPCell(input_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.u = None  # state

    def reset_states(self, batch_size, device):
        self.u = torch.zeros(batch_size, self.hidden_dim, device=device)
        
    def detach_states(self):
        if self.u is not None:
            self.u = self.u.detach()

    def forward(self, x):
        if self.u is None or self.u.size(0) != x.size(0) or self.u.device != x.device:
            self.reset_states(x.size(0), x.device)
            
        seq_len = x.size(1)
        h = None
        for t in range(seq_len):
            h, self.u = self.cell(x[:, t, :], self.u)
            
        out = self.fc(h)
        return out

def build_model(input_dim, units):
    return SequenceLSTMSNP(input_dim=input_dim, hidden_dim=units, output_dim=1)


### Output Layer
Simple `nn.Linear(hidden_dim, 1)` — no fuzzy logic.

### Build Model

In [32]:
# Quick model check
model = build_model(input_dim=1, units=8)
print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters())}")

SequenceLSTMSNP(
  (cell): LSTMSNPCell(
    (kernel): Linear(in_features=1, out_features=32, bias=False)
    (recurrent_kernel): Linear(in_features=8, out_features=32, bias=False)
  )
  (fc): Linear(in_features=8, out_features=1, bias=True)
)
Total params: 329


## Data Pipeline — Dow Jones Industrial Index

In [33]:
series = pd.read_csv(
    '/kaggle/input/datasets/satabartosarkar123/monthly-closings-of-the-dowjones-csv/monthly-closings-of-the-dowjones.csv',
    header=0,
    parse_dates=[0],
    index_col=0
)
raw_values = series.values.flatten()

In [34]:
# ============================================================
# 1. Load Time Series Data
# ============================================================
series = pd.read_csv(
    '/kaggle/input/datasets/satabartosarkar123/monthly-closings-of-the-dowjones-csv/monthly-closings-of-the-dowjones.csv',
    header=0,
    parse_dates=[0],
    index_col=0
)
raw_values = series.values.flatten()
print(f"Data shape: {raw_values.shape}")
print(f"First 5 values: {raw_values[:5]}")

Data shape: (291,)
First 5 values: [3645.  3626.  3634.  3620.5 3607. ]


In [35]:
# ============================================================
# 2. First-Order Differencing
# ============================================================

def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)

diff_values = difference(raw_values, 1)

In [36]:
# ============================================================
# 3. Convert to Supervised Learning Format (lag=1)
# ============================================================

def timeseries_to_supervised(data, lag=1):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag+1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values

supervised = timeseries_to_supervised(diff_values, 1)
print(f"Supervised data shape: {supervised.shape}")

Supervised data shape: (290, 2)


In [37]:
# ============================================================
# 4. Train-Validation-Test Split (Last 60 points as Test)
# ============================================================
n = len(supervised)
test_size = 60
train_val = supervised[:n - test_size]
test = supervised[n - test_size:]

val_size = int(len(train_val) * 0.1)
train = train_val[:-val_size]
val = train_val[-val_size:]

print(f"Train: {train.shape}, Val: {val.shape}, Test: {test.shape}")

# ============================================================
# 5. Feature Scaling
# ============================================================
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(train)  # fit only on train to avoid leakage
train_scaled = scaler.transform(train)
val_scaled = scaler.transform(val)
test_scaled = scaler.transform(test)

Train: (207, 2), Val: (23, 2), Test: (60, 2)


In [38]:
# ============================================================
# 6. Reshape for RNN Input
# ============================================================

X_train, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))

X_test, y_test = test_scaled[:, 0:-1], test_scaled[:, -1]
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (207, 1, 1), y_train shape: (207,)


In [39]:
#Gaussian noise injection (Eq. 29–30)
ref_std = X_train.std()
print(f"Reference std(x) used for noise scaling (train inputs): {ref_std:.6f}")

NOISE_LEVELS = [0.005, 0.05, 0.10]  # 0.5%, 5%, 10%

def add_gaussian_noise(X, noise_level, ref_std, seed=None):
    """
    x_noisy(t) = x(t) + eps(t),  eps(t) ~ N(0, sigma_eps^2),  sigma_eps = noise_level * std(x)
    Applied to input features X only — never to targets/labels.
    """
    rng = np.random.default_rng(seed)
    sigma_eps = noise_level * ref_std
    eps = rng.normal(loc=0.0, scale=sigma_eps, size=X.shape)
    return X + eps

Reference std(x) used for noise scaling (train inputs): 0.295900


In [40]:
#Evaluation function

def evaluate_on_test(model, X_test_eval, train, train_scaled, raw_values, scaler, device):
    """
    Warms up the model's recurrent state on clean training data, then runs
    single-step-ahead prediction on X_test_eval (clean or Gaussian-noise-corrupted
    input features). Test targets are always the clean, ground-truth values.
    """
    model.eval()
    model.reset_states(1, device)

    with torch.no_grad():
        for i in range(len(train_scaled)):
            X_raw = train_scaled[i, 0:-1]
            X_input = torch.tensor(X_raw, dtype=torch.float32).view(1, len(X_raw), 1).to(device)
            model(X_input)

        predictions = []
        for i in range(len(X_test_eval)):
            X = X_test_eval[i]
            X_input = torch.tensor(X, dtype=torch.float32).view(1, len(X), 1).to(device)
            yhat = model(X_input).item()

            new_row = [x for x in X] + [yhat]
            array = np.array(new_row).reshape(1, len(new_row))
            inverted = scaler.inverse_transform(array)[0, -1]
            inverted = inverted + raw_values[len(train) + i]
            predictions.append(inverted)

    actual = raw_values[-len(X_test_eval):]
    mse = mean_squared_error(actual, predictions)
    rmse = sqrt(mse)
    meanV = np.mean(actual)
    dominator = np.linalg.norm(np.array(actual) - meanV, 2)   # FIX: was predictions - meanV
    nmse = mse / np.power(dominator, 2)
    return predictions, rmse, mse, nmse

## Training Loop

In [41]:
import os

CHECKPOINT_DIR = "checkpoints_DOW_JONES"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def _ckpt_path(run):
    return os.path.join(CHECKPOINT_DIR, f"run_{run}.pt")

def _summary_path():
    return os.path.join(CHECKPOINT_DIR, "summary.pt")

all_rmse = []
all_mse = []
all_nmse = []
all_predictions = []
all_losses = []
all_models = []  # keep every trained model for later noise-robustness eval

print(f"\n--- [PyTorch] RUNNING ON {device} ---\n")
print(f"Checkpoints will be read/written under: {os.path.abspath(CHECKPOINT_DIR)}\n")

N_RUNS = 60
val_X = val_scaled[:, 0:-1]
PATIENCE = 10

summary_path = _summary_path()
if os.path.exists(summary_path):
    print("Found completed checkpoint summary -> loading, skipping training.")
    saved = torch.load(summary_path, map_location=device, weights_only=False)

    all_models = []
    for sd in saved['model_state_dicts']:
        m = build_model(input_dim=1, units=8).to(device)
        m.load_state_dict(sd)
        m.eval()
        all_models.append(m)

    all_rmse = saved['rmse']
    all_mse = saved['mse']
    all_nmse = saved['nmse']
    all_predictions = saved['predictions']
    all_losses = saved['losses']
    print(f"Loaded {len(all_models)} models from checkpoint.")

else:
    for run in range(N_RUNS):
        run_ckpt_path = _ckpt_path(run)
        if os.path.exists(run_ckpt_path):
            print(f'\n===== RUN {run + 1}/{N_RUNS} (resumed from checkpoint) =====')
            ckpt = torch.load(run_ckpt_path, map_location=device, weights_only=False)

            model = build_model(input_dim=1, units=8).to(device)
            model.load_state_dict(ckpt['model_state_dict'])
            model.eval()

            all_losses.append(ckpt['run_losses'])
            all_models.append(model)
            all_rmse.append(ckpt['rmse'])
            all_mse.append(ckpt['mse'])
            all_nmse.append(ckpt['nmse'])
            all_predictions.append(ckpt['predictions'])
            print(f"Loaded run {run + 1} -- RMSE: {ckpt['rmse']:.6f}, MSE: {ckpt['mse']:.6f}, NMSE: {ckpt['nmse']:.10f} "
                  f"(best val RMSE: {ckpt['best_val_rmse']:.6f})")
            continue

        print(f'\n===== RUN {run + 1}/{N_RUNS} =====')

        np.random.seed(run)
        torch.manual_seed(run)
        model = build_model(input_dim=1, units=8).to(device)
        # NOTE: no LSTM-SNP-style gate-bias init here -- PureGRU has no
        # .cell/.hidden_size to slice into; nn.GRU's own default init is used.

        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()
        run_losses = []

        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
        n_samples = X_train_t.size(0)

        best_val_rmse = float('inf')
        best_state = None
        patience_ctr = 0

        for epoch in range(100):
            model.train()
            model.reset_states(1, device)
            epoch_loss = 0.0

            for i in range(n_samples):
                x_i = X_train_t[i:i + 1]
                y_i = y_train_t[i:i + 1]

                optimizer.zero_grad()
                pred = model(x_i)
                loss = criterion(pred.squeeze(-1), y_i)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                model.detach_states()
                epoch_loss += loss.item()

            avg_loss = epoch_loss / n_samples
            run_losses.append(avg_loss)

            _, val_rmse, _, _ = evaluate_on_test(model, val_X, train, train_scaled, raw_values, scaler, device)
            print(f"Epoch {epoch + 1}/100 — train loss: {avg_loss:.6f}  val RMSE: {val_rmse:.6f}")

            if val_rmse < best_val_rmse:
                best_val_rmse = val_rmse
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"Early stopping at epoch {epoch + 1} (best val RMSE: {best_val_rmse:.6f})")
                    break

        model.load_state_dict(best_state)
        model.reset_states(1, device)

        all_losses.append(run_losses)
        all_models.append(model)
        print(f'Training complete for run {run + 1} (best val RMSE: {best_val_rmse:.6f})')

        predictions, rmse, mse, nmse = evaluate_on_test(
            model, X_test, train, train_scaled, raw_values, scaler, device
        )

        all_rmse.append(rmse)
        all_mse.append(mse)
        all_nmse.append(nmse)
        all_predictions.append(predictions)

        print(f'Run {run + 1} — RMSE: {rmse:.6f}, MSE: {mse:.6f}, NMSE: {nmse:.10f}')

        torch.save({
            'model_state_dict': model.state_dict(),
            'run_losses': run_losses,
            'rmse': rmse,
            'mse': mse,
            'nmse': nmse,
            'predictions': predictions,
            'best_val_rmse': best_val_rmse,
        }, run_ckpt_path)

    torch.save({
        'rmse': all_rmse,
        'mse': all_mse,
        'nmse': all_nmse,
        'predictions': all_predictions,
        'losses': all_losses,
        'model_state_dicts': [m.state_dict() for m in all_models],
    }, summary_path)


--- [PyTorch] RUNNING ON cuda ---

Checkpoints will be read/written under: /kaggle/working/checkpoints_DOW_JONES


===== RUN 1/60 =====
Epoch 1/100 — train loss: 0.107010  val RMSE: 159.688382
Epoch 2/100 — train loss: 0.090361  val RMSE: 159.646208
Epoch 3/100 — train loss: 0.090044  val RMSE: 159.576189
Epoch 4/100 — train loss: 0.089849  val RMSE: 159.519859
Epoch 5/100 — train loss: 0.089730  val RMSE: 159.482592
Epoch 6/100 — train loss: 0.089659  val RMSE: 159.460743
Epoch 7/100 — train loss: 0.089620  val RMSE: 159.452970
Epoch 8/100 — train loss: 0.089601  val RMSE: 159.454484
Epoch 9/100 — train loss: 0.089595  val RMSE: 159.459020
Epoch 10/100 — train loss: 0.089596  val RMSE: 159.464031
Epoch 11/100 — train loss: 0.089601  val RMSE: 159.467762
Epoch 12/100 — train loss: 0.089607  val RMSE: 159.469130
Epoch 13/100 — train loss: 0.089614  val RMSE: 159.467543
Epoch 14/100 — train loss: 0.089620  val RMSE: 159.462768
Epoch 15/100 — train loss: 0.089621  val RMSE: 159.454929
Ep

In [42]:
#Noise-robustness sweep (averaged over multiple noise draws, with explicit clean baseline)

N_NOISE_DRAWS = 10  # average over multiple ε(t) realizations per model, per level

# Re-evaluate the clean (0% noise) baseline through evaluate_on_test, so it
# uses the exact same evaluation path as the noisy conditions below.
clean_rmse, clean_mse, clean_nmse = [], [], []
for model in all_models:
    _, rmse, mse, nmse = evaluate_on_test(model, X_test, train, train_scaled, raw_values, scaler, device)
    clean_rmse.append(rmse)
    clean_mse.append(mse)
    clean_nmse.append(nmse)

print(f"[{'no noise':>10}]  "
      f"RMSE: {np.mean(clean_rmse):.6f} ± {np.std(clean_rmse):.6f}  |  "
      f"MSE: {np.mean(clean_mse):.6f} ± {np.std(clean_mse):.6f}  |  "
      f"NMSE: {np.mean(clean_nmse):.10f} ± {np.std(clean_nmse):.10f}")

noise_robustness = {}  # noise_level -> dict of per-model RMSE/MSE/NMSE (averaged over draws)

for noise_level in NOISE_LEVELS:
    level_rmse, level_mse, level_nmse = [], [], []

    for run, model in enumerate(all_models):
        draw_rmse, draw_mse, draw_nmse = [], [], []
        for draw in range(N_NOISE_DRAWS):
            seed = hash((noise_level, run, draw)) % (2 ** 32)
            X_test_noisy = add_gaussian_noise(X_test, noise_level, ref_std, seed=seed)
            _, rmse, mse, nmse = evaluate_on_test(
                model, X_test_noisy, train, train_scaled, raw_values, scaler, device
            )
            draw_rmse.append(rmse)
            draw_mse.append(mse)
            draw_nmse.append(nmse)

        level_rmse.append(np.mean(draw_rmse))  # this model's average over noise draws
        level_mse.append(np.mean(draw_mse))
        level_nmse.append(np.mean(draw_nmse))

    noise_robustness[noise_level] = {'rmse': level_rmse, 'mse': level_mse, 'nmse': level_nmse}
    print(f"[{noise_level * 100:5.1f}% noise]  "
          f"RMSE: {np.mean(level_rmse):.6f} ± {np.std(level_rmse):.6f}  |  "
          f"MSE: {np.mean(level_mse):.6f} ± {np.std(level_mse):.6f}  |  "
          f"NMSE: {np.mean(level_nmse):.10f} ± {np.std(level_nmse):.10f}")

[  no noise]  RMSE: 99.185833 ± 1.115079  |  MSE: 9839.072859 ± 214.202530  |  NMSE: 0.0455600703 ± 0.0009918701
[  0.5% noise]  RMSE: 99.185745 ± 1.115066  |  MSE: 9839.055410 ± 214.199802  |  NMSE: 0.0455599895 ± 0.0009918575
[  5.0% noise]  RMSE: 99.186272 ± 1.115056  |  MSE: 9839.159938 ± 214.197880  |  NMSE: 0.0455604735 ± 0.0009918486
[ 10.0% noise]  RMSE: 99.186370 ± 1.115179  |  MSE: 9839.180071 ± 214.222791  |  NMSE: 0.0455605667 ± 0.0009919639


In [46]:
import shutil
from IPython.display import FileLink

# 1. Zip the entire output directory
shutil.make_archive('kaggle_outputs', 'zip', '/kaggle/working')

# 2. Generate a clickable download link
FileLink(r'kaggle_outputs.zip')


/kaggle/working/kaggle_outputs.zip